# Third-Generation CALPHAD: Einstein Oscillator

This example demonstrates how to use the `GroundStateNode` and `EinsteinNode` to build a physically consistent thermodynamic model for a pure phase that obeys the Third Law of Thermodynamics ($C_p \to 0$ as $T \to 0$ K). We construct pure Copper (Cu) and Aluminum (Al) phases and use `jax.grad` to extract their Heat Capacities.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('../../zgraph/src'))
sys.path.append(os.path.abspath('../../thermograph/src'))
sys.path.append(os.path.abspath('../../'))

import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from zgraph import *
from thermograph.nodes.einstein import GroundStateNode, EinsteinNode
from external_data.einstein_params import CU_FCC, AL_FCC

## 1. Defining the ZGraph Architecture
We declare the independent variable $T$ and instantiate our purely normalized nodes.

In [ ]:
T = SignalNodes(0)

# Ground State Energies
cu_e0 = GroundStateNode(CU_FCC["E_0"]).compile_zgraph_engine()
al_e0 = GroundStateNode(AL_FCC["E_0"]).compile_zgraph_engine()

# 1-DOF Einstein Oscillators
cu_osc = EinsteinNode(CU_FCC["theta_acoustic"], T_index=0).compile_zgraph_engine()
al_osc = EinsteinNode(AL_FCC["theta_acoustic"], T_index=0).compile_zgraph_engine()


## 2. PGM Topology: Connecting the Leaves
We use a `FactorNode` to apply the structural weights. A pure monatomic lattice has 3 degrees of freedom, so we scale the 1-DOF oscillator by a factor of 3 in the connection matrix.

In [ ]:
# The weight matrix M applies [1.0 * E_0, 3.0 * Oscillator]
cu_phase = FactorNode(jnp.array([[1.0, 3.0]]), [cu_e0, cu_osc], beta=0.0)
al_phase = FactorNode(jnp.array([[1.0, 3.0]]), [al_e0, al_osc], beta=0.0)



## 3. Deriving Heat Capacity
Using JAX, we can automatically derive the heat capacity $C_p = -T \frac{\partial^2 G}{\partial T^2}$. Because the graph is fully differentiable across all temperature ranges, we can evaluate it directly down to absolute zero.

In [ ]:
# Compute dG/dT (Entropy is -dG/dT)
cu_grad = jax.grad(lambda t: cu_phase(jnp.atleast_1d(t)))
al_grad = jax.grad(lambda t: al_phase(jnp.atleast_1d(t)))

# Compute d2G/dT2
cu_grad2 = jax.grad(cu_grad)
al_grad2 = jax.grad(al_grad)

# Vectorize the functions for batch evaluation
cu_cp = jax.vmap(lambda t: -t * cu_grad2(t))
al_cp = jax.vmap(lambda t: -t * al_grad2(t))

T_vals = jnp.linspace(0.1, 1000, 500)
cp_cu_vals = cu_cp(T_vals)
cp_al_vals = al_cp(T_vals)


## 4. Visualizing Physical Consistency
Plotting the Heat Capacity reveals the hallmark signature of the Einstein model: unlike polynomial fits which often incorrectly predict finite or negative heat capacities at 0 K, the oscillator model smoothly and correctly forces $C_p \to 0$ as $T \to 0$.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=T_vals, y=cp_cu_vals, mode='lines', name='Cu (FCC)', line=dict(color='orange', width=3)))
fig.add_trace(go.Scatter(x=T_vals, y=cp_al_vals, mode='lines', name='Al (FCC)', line=dict(color='blue', width=3)))

fig.update_layout(
    title='Heat Capacity (Einstein Model)',
    xaxis_title='Temperature (K)',
    yaxis_title='Cp (J / mol K)',
    width=800,
    height=500
)
fig.show()
